# PolicyLens RAG v2.0 — Bilingual Cross-Encoder Reranker Fine-Tuning (Colab / Kaggle T4 GPU)

This notebook demonstrates the offline GPU training + ONNX CPU deployment workflow for **PolicyLens RAG** (`bachiep/policylens-rag`):

1. **Hard-Negative Mining:** Mine high-overlap distractor chunks from bilingual Vietnamese/English corporate policies (`data/sample/*.md`) using Hybrid Dense (`paraphrase-multilingual-MiniLM-L12-v2`) + BM25 retrieval.
2. **Leak-Free Evaluation Split:** Hold out unseen queries to measure genuine generalization (`MRR@10`, `HitRate@5`).
3. **AMP `fp16` Cross-Encoder Training on T4 GPU:** Fine-tune `cross-encoder/mmarco-mMiniLMv2-L12-H384-v1` using PyTorch Automatic Mixed Precision (`fp16`) in `sentence-transformers`.
4. **Before vs. After Benchmark:** Compare zero-shot Cross-Encoder vs. domain-adapted Cross-Encoder.
5. **ONNX Export & INT8 Quantization:** Export and quantize the fine-tuned reranker to CPU-ready ONNX (`onnxruntime`) for lightweight production serving without PyTorch at runtime.

In [ ]:
# Run on Google Colab / Kaggle with a T4 GPU runtime (Change runtime type -> T4 GPU)
!nvidia-smi
!pip install -q "sentence-transformers>=3.2" "datasets>=2.20" \
    "accelerate>=0.33" "optimum[onnxruntime]>=1.22" "onnxruntime>=1.19" fastembed rank-bm25

In [ ]:
import json
import os
from pathlib import Path

import numpy as np
import torch

print(f"PyTorch: {torch.__version__} | CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_FP16 = torch.cuda.is_available()

REPO_DIR = Path("policylens-rag")
if not Path("data/sample").exists() and not REPO_DIR.exists():
    !git clone -q https://github.com/bachiep/policylens-rag.git
if REPO_DIR.exists():
    os.chdir(REPO_DIR)

print("Working directory:", Path.cwd())

## 1. Load Bilingual Corpus & Chunk Policy Documents

In [ ]:
import sys

sys.path.insert(0, str(Path("src").resolve()))

from policylens.ingestion.chunker import MarkdownRecursiveChunker
from policylens.ingestion.pipeline import load_corpus_documents

docs = load_corpus_documents(Path("data/sample"))
chunker = MarkdownRecursiveChunker(chunk_size=900, chunk_overlap=150)
all_chunks = [chunk for doc in docs for chunk in chunker.chunk_document(doc)]
chunks_by_id = {c.chunk_id: c for c in all_chunks}

eval_rows = [
    json.loads(line)
    for line in Path("evals/dataset.jsonl").read_text(encoding="utf-8").splitlines()
    if line.strip()
]
answerable_rows = [r for r in eval_rows if not r["is_unanswerable"]]
print(
    f"Loaded {len(docs)} policy docs -> {len(all_chunks)} chunks | "
    f"{len(answerable_rows)} answerable queries"
)

## 2. Hard-Negative Mining from Hybrid Dense + BM25 Retrieval

We split the 28 answerable queries into 20 training queries and 8 held-out evaluation queries (stratified across VI/EN and categories). For each query, we retrieve Top-12 candidates with `fastembed` (`paraphrase-multilingual-MiniLM-L12-v2`) + `BM25Okapi` RRF and select high-ranking non-ground-truth chunks as **hard negatives** (including intra-document adjacent sections and cross-document semantic distractors).

In [ ]:
from fastembed import TextEmbedding

from policylens.retrieval.bm25 import BM25Index

EMBED_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
dense_model = TextEmbedding(model_name=EMBED_MODEL)
chunk_texts = [c.retrieval_text() for c in all_chunks]
chunk_vecs = np.vstack([
    vec / max(np.linalg.norm(vec), 1e-8)
    for vec in dense_model.passage_embed(chunk_texts)
])

bm25 = BM25Index(all_chunks)

def mine_candidates(query: str, top_n: int = 12) -> list[str]:
    q_vec = next(dense_model.query_embed([query]))
    q_vec = q_vec / max(np.linalg.norm(q_vec), 1e-8)
    sims = chunk_vecs @ q_vec
    dense_order = [all_chunks[i].chunk_id for i in np.argsort(-sims)[:top_n]]
    sparse_order = [hit.chunk_id for hit in bm25.search(query, top_k=top_n)]

    rrf = {}
    for rank, cid in enumerate(dense_order, start=1):
        rrf[cid] = rrf.get(cid, 0.0) + 1.0 / (60 + rank)
    for rank, cid in enumerate(sparse_order, start=1):
        rrf[cid] = rrf.get(cid, 0.0) + 1.0 / (60 + rank)
    return [cid for cid, _ in sorted(rrf.items(), key=lambda x: x[1], reverse=True)[:top_n]]

train_queries = [q for idx, q in enumerate(answerable_rows) if idx % 7 not in (2, 5)]
val_queries = [q for idx, q in enumerate(answerable_rows) if idx % 7 in (2, 5)]

train_pairs = []
for item in train_queries:
    pos_ids = set(item["expected_chunk_ids"])
    candidates = mine_candidates(item["query"], top_n=12)
    for pid in pos_ids:
        train_pairs.append({
            "query": item["query"],
            "passage": chunks_by_id[pid].retrieval_text(),
            "label": 1.0,
        })
    hard_negs = [cid for cid in candidates if cid not in pos_ids][:4]
    for nid in hard_negs:
        train_pairs.append({
            "query": item["query"],
            "passage": chunks_by_id[nid].retrieval_text(),
            "label": 0.0,
        })

print(
    f"Train queries: {len(train_queries)} -> {len(train_pairs)} pairs | "
    f"Val queries: {len(val_queries)}"
)

## 3. Baseline Cross-Encoder Evaluation (`MRR@10` & `HitRate@5` Before Fine-Tuning)

In [ ]:
from sentence_transformers import CrossEncoder

BASE_RERANKER = "cross-encoder/mmarco-mMiniLMv2-L12-H384-v1"
baseline_model = CrossEncoder(BASE_RERANKER, num_labels=1, max_length=384, device=DEVICE)

def eval_reranker(model: CrossEncoder, queries: list[dict]) -> dict[str, float]:
    mrrs, hits5 = [], []
    for item in queries:
        candidates = mine_candidates(item["query"], top_n=12)
        pairs = [(item["query"], chunks_by_id[cid].retrieval_text()) for cid in candidates]
        with torch.amp.autocast(device_type="cuda", enabled=USE_FP16):
            scores = model.predict(pairs, show_progress_bar=False)
        scored = sorted(zip(candidates, scores, strict=True), key=lambda x: x[1], reverse=True)
        ranked = [cid for cid, _ in scored]
        expected = set(item["expected_chunk_ids"])
        hits5.append(1.0 if expected & set(ranked[:5]) else 0.0)
        rr = next((1.0 / r for r, cid in enumerate(ranked[:10], start=1) if cid in expected), 0.0)
        mrrs.append(rr)
    return {"MRR@10": float(np.mean(mrrs)), "HitRate@5": float(np.mean(hits5))}

before_metrics = eval_reranker(baseline_model, val_queries)
print("Before Fine-Tuning (Zero-Shot):", before_metrics)

## 4. Fine-Tune Cross-Encoder with PyTorch AMP (`fp16`) on T4 GPU

In [ ]:
from datasets import Dataset
from sentence_transformers.cross_encoder import CrossEncoderTrainer, CrossEncoderTrainingArguments
from sentence_transformers.cross_encoder.losses import BinaryCrossEntropyLoss

OUTPUT_DIR = Path("artifacts/policylens-reranker-ft")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

train_dataset = Dataset.from_list(train_pairs)
ft_model = CrossEncoder(BASE_RERANKER, num_labels=1, max_length=384, device=DEVICE)
loss = BinaryCrossEntropyLoss(model=ft_model, pos_weight=torch.tensor([3.0], device=DEVICE))

training_args = CrossEncoderTrainingArguments(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=6,
    per_device_train_batch_size=16,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    fp16=USE_FP16,
    logging_steps=10,
    save_strategy="no",
    report_to="none",
)

trainer = CrossEncoderTrainer(
    model=ft_model,
    args=training_args,
    train_dataset=train_dataset,
    loss=loss,
)
trainer.train()
ft_model.save_pretrained(str(OUTPUT_DIR))

after_metrics = eval_reranker(ft_model, val_queries)
print(
    f"Before MRR@10: {before_metrics['MRR@10']:.4f} | "
    f"After MRR@10: {after_metrics['MRR@10']:.4f}"
)
print(
    f"Before HitRate@5: {before_metrics['HitRate@5']:.4f} | "
    f"After HitRate@5: {after_metrics['HitRate@5']:.4f}"
)

## 5. Export to ONNX + INT8 Dynamic Quantization for CPU Serving

Export the fine-tuned PyTorch checkpoint via HuggingFace Optimum and apply INT8 dynamic quantization for fast, zero-PyTorch CPU inference inside `policylens-rag`.

In [ ]:
import time

import onnxruntime as ort
from onnxruntime.quantization import QuantType, quantize_dynamic
from optimum.onnxruntime import ORTModelForSequenceClassification
from transformers import AutoTokenizer

ONNX_DIR = Path("artifacts/policylens-reranker-onnx")
ort_model = ORTModelForSequenceClassification.from_pretrained(str(OUTPUT_DIR), export=True)
tokenizer = AutoTokenizer.from_pretrained(str(OUTPUT_DIR))
ort_model.save_pretrained(str(ONNX_DIR))
tokenizer.save_pretrained(str(ONNX_DIR))

fp32_onnx = ONNX_DIR / "model.onnx"
int8_onnx = ONNX_DIR / "model_int8.onnx"
quantize_dynamic(
    model_input=str(fp32_onnx),
    model_output=str(int8_onnx),
    weight_type=QuantType.QInt8,
)

fp32_mb = fp32_onnx.stat().st_size / (1024 * 1024)
int8_mb = int8_onnx.stat().st_size / (1024 * 1024)
print(f"ONNX Size Reduction: {fp32_mb:.1f} MB (FP32) -> {int8_mb:.1f} MB (INT8)")

# Smoke-test CPU ONNXRuntime inference latency
session = ort.InferenceSession(str(int8_onnx), providers=["CPUExecutionProvider"])
inputs = tokenizer(
    ["FIN-POL-03 phụ cấp ăn uống công tác nội địa là bao nhiêu?"],
    [chunks_by_id["FIN-POL-03#c02"].retrieval_text()],
    padding=True,
    truncation=True,
    max_length=384,
    return_tensors="np",
)
ort_inputs = {k: v for k, v in inputs.items() if k in {i.name for i in session.get_inputs()}}
t0 = time.perf_counter()
logits = session.run(None, ort_inputs)[0]
cpu_ms = (time.perf_counter() - t0) * 1000
prob = float(1.0 / (1.0 + np.exp(-logits[0][0])))
print(f"CPU INT8 ONNX Reranker Score: {prob:.4f} in {cpu_ms:.2f} ms")